# Section 6.1 — LangGraph Agents: Travel Edition

Four agents, each showing a different LangGraph pattern, all built around travel:

| Agent | Pattern | Travel use case |
|---|---|---|
| 1 — Destination Intelligence | Agent → Tool → Agent loop | Researches a destination: facts, flights, hotels, total cost |
| 2 — Travel Request Router | Classify → route to a specialist | **Flight booking**, **hotel booking** or **itinerary planner** agent |
| 3 — Multi-City Trip Planner | Quality gate with a loop | Plans several cities, scores the plan, retries until score >= 7 |
| 4 — Trip Approval Pipeline | Human-in-the-loop pause | Research → Plan → Quality loop → Human pause → Finalise |

Uses `MemorySaver` for the checkpoint, which works with all langgraph versions.

> All travel data in this notebook (fares, hotel prices, availability) is **sample data** for learning, the same way the tools in a tutorial return fixed values.

---
## Step 1 — Install

In [1]:
!pip install -q \
    "langgraph>=0.2.0" \
    "langchain==0.2.16" \
    "langchain-openai==0.1.23" \
    "langchain-core==0.2.43" \
    "openai==1.51.0" \
    "httpx==0.27.2"

print("Done!")

Done!


---
## Step 2 — API Key, Imports and Color Printer

Set your key in the environment (`export OPENAI_API_KEY=...`) or paste it into the variable below.
**Never commit a real key to a notebook you share.**

In [2]:
import os, warnings, logging, random, json, re
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

# Paste your key here, or set the OPENAI_API_KEY environment variable before starting the notebook.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

from typing import TypedDict, List, Annotated
import operator
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver   # MemorySaver works with all langgraph versions

# Color printer
class C:
    AGENT  = "\033[1m\033[94m"
    TOOL   = "\033[96m"
    RESULT = "\033[93m"
    WRITE  = "\033[35m"
    READ   = "\033[34m"
    ROUTE  = "\033[33m"
    DONE   = "\033[1m\033[92m"
    PAUSE  = "\033[1m\033[91m"
    HUMAN  = "\033[1m\033[95m"
    END    = "\033[0m"

def agent(msg):  print(f"{C.AGENT}  [AGENT]  {C.END}{msg}")
def tool_c(msg): print(f"{C.TOOL}  [TOOL]   {C.END}{msg}")
def result(msg): print(f"{C.RESULT}  [RESULT] {C.END}{msg}")
def writes(msg): print(f"{C.WRITE}  [WRITES] {C.END}{msg}")
def reads(msg):  print(f"{C.READ}  [READS]  {C.END}{msg}")
def route(msg):  print(f"{C.ROUTE}  [ROUTE]  {C.END}{msg}")
def done(msg):   print(f"{C.DONE}  [OUTPUT] {C.END}{msg}")
def pause(msg):  print(f"{C.PAUSE}  [PAUSE]  {C.END}{msg}")
def human(msg):  print(f"{C.HUMAN}  [HUMAN]  {C.END}{msg}")

print("Ready!")
agent("Agent thinking")
tool_c("Tool called")
result("Tool result")
writes("Agent writes to state")
reads("Agent reads from state")
route("Routing decision")
done("Final output")
pause("Graph paused for human")
human("Human action")

Ready!
  [AGENT]  Agent thinking
  [TOOL]   Tool called
  [RESULT] Tool result
  [WRITES] Agent writes to state
  [READS]  Agent reads from state
  [ROUTE]  Routing decision
  [OUTPUT] Final output
  [PAUSE]  Graph paused for human
  [HUMAN]  Human action


### Choosing the language model

The notebook uses OpenAI `gpt-3.5-turbo` when a key is set and OpenAI is reachable.

If there is no key, or OpenAI cannot be reached (for example on a locked-down network), it falls back to a small **rule-based stand-in** that has the same interface, so every graph below still runs end to end. The stand-in writes templated text instead of real model text, and the next cell prints which one is active.

In [3]:
class OfflineTravelLLM:
    """Rule-based stand-in for ChatOpenAI. Used ONLY when OpenAI cannot be used.
    Same interface (invoke / bind_tools), so the graphs below run unchanged."""

    CITIES = ["mumbai", "delhi", "bengaluru", "goa", "jaipur", "kochi", "udaipur"]
    DAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

    def __init__(self, tools=None):
        self.tools = tools

    def bind_tools(self, tools):
        return OfflineTravelLLM(tools)

    # ---- helpers --------------------------------------------------
    def _cities_in(self, text):
        found = [(text.lower().find(c), c) for c in self.CITIES if c in text.lower()]
        return [c for _, c in sorted(found)]

    @staticmethod
    def _money(s):
        return int(s.replace(",", ""))

    # ---- tool-calling agent (Agent 1) -----------------------------
    def _tool_agent(self, messages):
        human_text = next(m.content for m in messages if isinstance(m, HumanMessage))
        city = re.search(r"report for (\w+)", human_text).group(1)
        origin, day, nights = re.search(r"from (\w+) on (\w+) for (\d+) nights", human_text).groups()
        calls = {}
        order = []
        for m in messages:
            if isinstance(m, AIMessage) and m.tool_calls:
                for tc in m.tool_calls:
                    calls[tc["id"]] = tc["name"]
                    order.append(tc["name"])
        results = {calls[m.tool_call_id]: m.content for m in messages
                   if isinstance(m, ToolMessage) and m.tool_call_id in calls}
        step = len(order)

        def call(name, args):
            return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": f"call_{step}"}])

        if step == 0:
            return call("get_destination_info", {"city": city})
        if step == 1:
            return call("search_flights", {"origin": origin, "destination": city, "date": day})
        if step == 2:
            return call("search_hotels", {"city": city, "nights": int(nights)})
        if step == 3:
            fare = self._money(re.search(r"Cheapest Rs ([\d,]+)", results["search_flights"]).group(1))
            hotel = self._money(re.search(r"Mid-range Rs [\d,]+/night \(Rs ([\d,]+) total\)", results["search_hotels"]).group(1))
            daily = self._money(re.search(r"avg Rs ([\d,]+)/day", results["get_destination_info"]).group(1))
            expr = f"{fare}*2 + {hotel} + {nights}*{daily}"
            return call("calculate", {"expression": expr})
        # final report
        info = results["get_destination_info"]
        flight = re.search(r"Cheapest (Rs [\d,]+) \(([^)]*)\)", results["search_flights"])
        hotel = re.search(r"Mid-range (Rs [\d,]+)/night \(Rs ([\d,]+) total\)", results["search_hotels"])
        total = f"{float(results['calculate'].replace('Result: ', '')):,.0f}"
        season = re.search(r"best ([^,]+),", info).group(1)
        attractions = info.split(", ")[1]
        daily = re.search(r"avg (Rs [\d,]+)/day", info).group(1)
        text = (
            f"### Trip Intelligence Report: {city.title()} from {origin.title()} ({nights} nights, {day})\n\n"
            f"**Destination facts:**\n- Best season: {season}\n- Highlights: {attractions}\n- Daily spend (excluding stay): {daily}\n\n"
            f"**Cheapest flight (one way):** {flight.group(1)} ({flight.group(2)})\n\n"
            f"**Mid-range hotel:** {hotel.group(1)} per night, Rs {hotel.group(2)} for {nights} nights\n\n"
            f"**Estimated total for one traveller:** Rs {total} "
            f"(return flight + hotel + {nights} days of daily spend).\n\n"
            f"Book the flight early for {day.title()} and check the season note above before you confirm dates."
        )
        return AIMessage(content=text)

    # ---- plain prompt (Agents 2, 3, 4) ----------------------------
    def _prompt_agent(self, p):
        low = p.lower()
        if p.startswith("Classify."):
            req = p.split("Request: ", 1)[1]
            cities = self._cities_in(req)
            if any(w in req.lower() for w in ["flight", "fly ", "airline"]):
                typ = "flight"
            elif any(w in req.lower() for w in ["hotel", "stay", "room", "resort"]):
                typ = "hotel"
            else:
                typ = "itinerary"
            if typ == "flight" and len(cities) >= 2:
                origin, city = cities[0], cities[1]
            else:
                dest = [c for c in cities if c not in ("mumbai", "delhi", "bengaluru")]
                city = dest[0] if dest else (cities[0] if cities else "none")
                origin = "mumbai"
            day = next((d for d in self.DAYS if d in req.lower()), "friday")
            m = re.search(r"(\d+)\s*nights?", req.lower())
            nights = int(m.group(1)) if m else 2
            return json.dumps({"type": typ, "city": city.title(), "origin": origin.title(),
                               "date": day.title(), "nights": nights})
        if "ONE number only" in p:
            return "8"
        if p.startswith("You are a flight booking agent"):
            opts = re.search(r"Flight options: (.*)", p).group(1)
            return (f"Here is what I found: {opts} "
                    "I can book the cheapest option for you right now, I only need the traveller's name.")
        if p.startswith("You are a hotel booking agent"):
            opts = re.search(r"Hotel options: (.*)", p).group(1)
            mid = re.search(r"Mid-range (Rs [\d,]+/night \(Rs [\d,]+ total\))", opts).group(1)
            return (f"The mid-range option at {mid} gives the best balance of price and location for your stay. "
                    "Shall I go ahead and book it, or would you like to see the premium option?")
        if p.startswith("You are an itinerary planner agent"):
            info = re.search(r"Destination: (.*)", p).group(1)
            city = info.split(":")[0]
            season = re.search(r"best ([^,]+),", info).group(1)
            attractions = info.split(", ")[1]
            return (f"Day 1 in {city}: start with {attractions}, keeping the afternoon free to rest. "
                    f"Day 2: take a slow morning and finish with an evening walk; {season} is the best time to visit.")
        if "write one-sentence plan for each city" in low or "concise travel plan for each city" in low:
            out = []
            for line in re.findall(r"^([A-Z]+): (.*)$", p, flags=re.M):
                name, body = line
                info = body.split(" | ")[0]
                extra = " | ".join(body.split(" | ")[1:])
                season = re.search(r"best ([^,]+),", info).group(1)
                attractions = info.split(", ")[1]
                if "rooms available" in extra:
                    avail = re.search(r"(\d+) of (\d+)", extra)
                    hint = f"lock in your dates now ({avail.group(1)} of {avail.group(2)} partner rooms are still free), then cover {attractions}"
                else:
                    mid = re.search(r"Mid-range (Rs [\d,]+)/night", extra)
                    hint = f"stay at a mid-range hotel ({mid.group(1)}/night) and cover {attractions}" if mid else f"cover {attractions}"
                out.append(f"{name}: In the {season} season, {hint}.")
            return "\n".join(out)
        return "OK."

    def invoke(self, messages):
        if isinstance(messages, str):
            return AIMessage(content=self._prompt_agent(messages))
        if self.tools is not None:
            return self._tool_agent(messages)
        return AIMessage(content=self._prompt_agent("\n".join(m.content for m in messages)))


def get_llm():
    """Use real OpenAI when possible, otherwise the offline stand-in."""
    if OPENAI_API_KEY:
        try:
            real = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, timeout=15, max_retries=0)
            real.invoke("ping")
            return real, "OpenAI gpt-3.5-turbo (real model)"
        except Exception as e:
            print("OpenAI not usable:", type(e).__name__, "-> using offline stand-in")
    return OfflineTravelLLM(), "OFFLINE rule-based stand-in (no OpenAI key or OpenAI not reachable)"

llm, LLM_MODE = get_llm()
print("LLM mode:", LLM_MODE)

LLM mode: OFFLINE rule-based stand-in (no OpenAI key or OpenAI not reachable)


---
## Step 3 — Define Real Tools

In [4]:
@tool
def get_destination_info(city: str) -> str:
    "Get destination data: best season, top attractions, average daily spend, local tips."
    cities = {
        "goa":     "Goa: best Nov-Feb, beaches + old Portuguese quarter, avg Rs 3,500/day excluding stay, monsoon Jun-Sep is off-season.",
        "jaipur":  "Jaipur: best Oct-Mar, Amber Fort + City Palace + bazaars, avg Rs 2,500/day excluding stay, summers cross 40C.",
        "kochi":   "Kochi: best Oct-Feb, Fort Kochi + backwater cruises, avg Rs 3,000/day excluding stay, heavy rain Jun-Aug.",
        "udaipur": "Udaipur: best Sep-Mar, City Palace + Lake Pichola, avg Rs 3,200/day excluding stay, sunset boat rides are popular.",
    }
    return cities.get(city.lower().strip(), "Destination not found. Try: goa, jaipur, kochi, udaipur")

@tool
def calculate(expression: str) -> str:
    "Calculate any math expression. Example: 4350 * 2 + 13500 or 3 * 3500"
    try: return "Result: " + str(round(eval(expression, {"__builtins__": {}}, {}), 2))
    except: return "Error in expression"

@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    "Search flights between two cities on a date. Returns the cheapest and the fastest option (one way)."
    base = {"goa": (4200, "1h25m"), "jaipur": (5800, "2h10m"), "kochi": (5200, "2h05m"), "udaipur": (5600, "1h35m")}
    fare, dur = base.get(destination.lower().strip(), (5000, "2h00m"))
    random.seed(len(origin) + len(destination) + len(date))
    cheapest = fare + random.randint(-300, 600)
    fastest = cheapest + 750
    return (f"{origin.title()} -> {destination.title()} on {date}: "
            f"Cheapest Rs {cheapest:,} (Budget Air, 07:15, {dur}); "
            f"Fastest Rs {fastest:,} (Prime Air, 09:40, {dur}).")

@tool
def search_hotels(city: str, nights: int) -> str:
    "Search hotels in a city. Returns budget, mid-range and premium options with the total cost for the given nights."
    rates = {"goa": (1800, 4500, 9500), "jaipur": (1500, 3800, 8800),
             "kochi": (1700, 4000, 8200), "udaipur": (2000, 4800, 10500)}
    if city.lower().strip() not in rates:
        return "City not found. Try: goa, jaipur, kochi, udaipur"
    b, m, p = rates[city.lower().strip()]
    return (f"{city.title()}, {nights} nights: "
            f"Budget Rs {b:,}/night (Rs {b*nights:,} total) | "
            f"Mid-range Rs {m:,}/night (Rs {m*nights:,} total) | "
            f"Premium Rs {p:,}/night (Rs {p*nights:,} total).")

@tool
def check_availability(city: str, date: str) -> str:
    "Check how many partner hotel rooms are available in a city on a given date."
    random.seed(len(city) + len(date))
    base = {"goa": 40, "jaipur": 35, "kochi": 30, "udaipur": 25}
    b = base.get(city.lower(), 30)
    return f"{b - random.randint(2,8)} of {b} partner rooms available in {city} on {date}."

@tool
def get_local_transport_cost(distance_km: float, vehicle_type: str) -> str:
    "Calculate local transport cost. vehicle_type: scooter or taxi."
    rate = 4 if vehicle_type.lower() == "scooter" else 14
    cost = round(distance_km * rate, 2)
    return f"Local transport for {distance_km}km by {vehicle_type}: Rs {cost}"

tools = [get_destination_info, calculate, search_flights, search_hotels, check_availability, get_local_transport_cost]
llm_with_tools = llm.bind_tools(tools)
tool_map = {t.name: t for t in tools}

print("Tools:", [t.name for t in tools])
print(get_destination_info.invoke({"city": "goa"}))
print(check_availability.invoke({"city": "udaipur", "date": "Friday"}))
print(search_flights.invoke({"origin": "Mumbai", "destination": "Goa", "date": "Friday"}))
print(search_hotels.invoke({"city": "jaipur", "nights": 3}))
print(get_local_transport_cost.invoke({"distance_km": 12, "vehicle_type": "taxi"}))

Tools: ['get_destination_info', 'calculate', 'search_flights', 'search_hotels', 'check_availability', 'get_local_transport_cost']
Goa: best Nov-Feb, beaches + old Portuguese quarter, avg Rs 3,500/day excluding stay, monsoon Jun-Sep is off-season.
21 of 25 partner rooms available in udaipur on Friday.
Mumbai -> Goa on Friday: Cheapest Rs 4,113 (Budget Air, 07:15, 1h25m); Fastest Rs 4,863 (Prime Air, 09:40, 1h25m).
Jaipur, 3 nights: Budget Rs 1,500/night (Rs 4,500 total) | Mid-range Rs 3,800/night (Rs 11,400 total) | Premium Rs 8,800/night (Rs 26,400 total).
Local transport for 12.0km by taxi: Rs 168.0


---
## Step 4 — Shared Tool Runner

In [5]:
def run_tools(state: dict) -> dict:
    last = state["messages"][-1]
    results = []
    for tc in last.tool_calls:
        tool_c(f"{tc['name']}({tc['args']})")
        res = tool_map[tc["name"]].invoke(tc["args"])
        result(str(res))
        results.append(ToolMessage(content=str(res), tool_call_id=tc["id"]))
    return {"messages": results}

print("Tool runner ready.")

Tool runner ready.


---
## Agent 1 — Destination Intelligence Agent

Uses tools to research a destination, finds a flight and a hotel, calculates the trip cost, writes the report.

**Pattern:** Agent → Tool → Agent → Tool → Agent → Done

In [6]:
class TripIntelState(TypedDict):
    messages: Annotated[List, operator.add]
    city: str
    report: str

def intel_agent(state: TripIntelState) -> dict:
    agent("Destination agent thinking...")
    response = llm_with_tools.invoke(state["messages"])
    if not (hasattr(response, "tool_calls") and response.tool_calls):
        writes("report → state")
        return {"messages": [response], "report": response.content}
    return {"messages": [response]}

def route_intel(state: TripIntelState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls: return "run_tools"
    return END

g1 = StateGraph(TripIntelState)
g1.add_node("intel_agent", intel_agent)
g1.add_node("run_tools",   run_tools)
g1.set_entry_point("intel_agent")
g1.add_conditional_edges("intel_agent", route_intel, {"run_tools": "run_tools", END: END})
g1.add_edge("run_tools", "intel_agent")
intel_app = g1.compile()

print("=" * 55)
print("Agent 1: Destination Intelligence — Goa")
print("=" * 55)
intel_result = intel_app.invoke({
    "city": "Goa", "report": "",
    "messages": [
        SystemMessage(content="You are a travel intelligence agent. Use get_destination_info, search_flights, search_hotels, and calculate tools. Write a brief trip intelligence report at the end."),
        HumanMessage(content="Give a full trip intelligence report for Goa, flying from Mumbai on Friday for 3 nights. Include destination facts, the cheapest flight, the mid-range hotel total, and the estimated total cost for one traveller (return flight = cheapest fare x 2, plus hotel, plus 3 days of daily spend).")
    ]
})
print("")
done("TRIP INTELLIGENCE REPORT:")
print(intel_result["report"])

Agent 1: Destination Intelligence — Goa
  [AGENT]  Destination agent thinking...
  [TOOL]   get_destination_info({'city': 'Goa'})
  [RESULT] Goa: best Nov-Feb, beaches + old Portuguese quarter, avg Rs 3,500/day excluding stay, monsoon Jun-Sep is off-season.
  [AGENT]  Destination agent thinking...
  [TOOL]   search_flights({'origin': 'Mumbai', 'destination': 'Goa', 'date': 'Friday'})
  [RESULT] Mumbai -> Goa on Friday: Cheapest Rs 4,113 (Budget Air, 07:15, 1h25m); Fastest Rs 4,863 (Prime Air, 09:40, 1h25m).
  [AGENT]  Destination agent thinking...
  [TOOL]   search_hotels({'city': 'Goa', 'nights': 3})
  [RESULT] Goa, 3 nights: Budget Rs 1,800/night (Rs 5,400 total) | Mid-range Rs 4,500/night (Rs 13,500 total) | Premium Rs 9,500/night (Rs 28,500 total).
  [AGENT]  Destination agent thinking...
  [TOOL]   calculate({'expression': '4113*2 + 13500 + 3*3500'})
  [RESULT] Result: 32226
  [AGENT]  Destination agent thinking...
  [WRITES] report → state

  [OUTPUT] TRIP INTELLIGENCE REPORT:
##

---
## Agent 2 — Travel Request Router

Classifies the request → routes to a specialist → the specialist uses its tool.

```
classify_node → flight_specialist    (uses search_flights)
             → hotel_specialist      (uses search_hotels)
             → itinerary_specialist  (uses get_destination_info)
```

In [7]:
class TravelRequestState(TypedDict):
    request: str
    request_type: str
    city_mentioned: str
    origin_mentioned: str
    date_mentioned: str
    nights_mentioned: int
    response: str

def classify_node(state: TravelRequestState) -> dict:
    import json as j
    reads("request from state")
    r = llm.invoke(
        'Classify. Reply JSON only: {"type":"flight|hotel|itinerary","city":"destination city or none",'
        '"origin":"departure city","date":"day or date","nights":2}\n\nRequest: '
        + state["request"]
    )
    try:
        data = j.loads(r.content)
        rtype = data.get("type", "itinerary")
        city, origin = data.get("city", "none"), data.get("origin", "Mumbai")
        date, nights = data.get("date", "Friday"), int(data.get("nights", 2))
    except:
        rtype, city, origin, date, nights = "itinerary", "none", "Mumbai", "Friday", 2
    if rtype not in ("flight", "hotel", "itinerary"): rtype = "itinerary"
    writes(f"request_type={rtype} | city={city} | origin={origin} | date={date} | nights={nights} → state")
    return {"request_type": rtype, "city_mentioned": city, "origin_mentioned": origin,
            "date_mentioned": date, "nights_mentioned": nights}

def route_request(state: TravelRequestState) -> str:
    route(f"request_type={state['request_type']} → {state['request_type']}_specialist")
    return state["request_type"] + "_specialist"

def flight_specialist(state: TravelRequestState) -> dict:
    reads("request_type + city + origin + date from state")
    options = ""
    if state["city_mentioned"] not in ["none", ""]:
        tool_c(f"search_flights({state['origin_mentioned']}, {state['city_mentioned']}, {state['date_mentioned']})")
        options = search_flights.invoke({"origin": state["origin_mentioned"],
                                         "destination": state["city_mentioned"],
                                         "date": state["date_mentioned"]})
        result(options[:120])
    r = llm.invoke("You are a flight booking agent.\nFlight options: " + options + "\nRequest: " + state["request"] + "\n2 sentences.")
    writes("response → state")
    return {"response": "FLIGHT BOOKING AGENT: " + r.content}

def hotel_specialist(state: TravelRequestState) -> dict:
    reads("request_type + city + nights from state")
    options = ""
    if state["city_mentioned"] not in ["none", ""]:
        tool_c(f"search_hotels({state['city_mentioned']}, {state['nights_mentioned']})")
        options = search_hotels.invoke({"city": state["city_mentioned"], "nights": state["nights_mentioned"]})
        result(options[:120])
    r = llm.invoke("You are a hotel booking agent.\nHotel options: " + options + "\nRequest: " + state["request"] + "\n2 sentences.")
    writes("response → state")
    return {"response": "HOTEL BOOKING AGENT: " + r.content}

def itinerary_specialist(state: TravelRequestState) -> dict:
    reads("request_type + city from state")
    info = ""
    if state["city_mentioned"] not in ["none", ""]:
        tool_c(f"get_destination_info({state['city_mentioned']})")
        info = get_destination_info.invoke({"city": state["city_mentioned"]})
        result(info[:120])
    r = llm.invoke("You are an itinerary planner agent.\nDestination: " + info + "\nRequest: " + state["request"] + "\n2 sentences.")
    writes("response → state")
    return {"response": "ITINERARY PLANNER AGENT: " + r.content}

g2 = StateGraph(TravelRequestState)
g2.add_node("classify_node",         classify_node)
g2.add_node("flight_specialist",     flight_specialist)
g2.add_node("hotel_specialist",      hotel_specialist)
g2.add_node("itinerary_specialist",  itinerary_specialist)
g2.set_entry_point("classify_node")
g2.add_conditional_edges("classify_node", route_request,
    {"flight_specialist":"flight_specialist","hotel_specialist":"hotel_specialist","itinerary_specialist":"itinerary_specialist"})
for n in ["flight_specialist","hotel_specialist","itinerary_specialist"]:
    g2.add_edge(n, END)
request_app = g2.compile()

for c in [
    "Book me a flight from Mumbai to Goa this Friday morning, cheapest option please.",
    "I need a hotel in Jaipur for 3 nights, mid-range with a good location.",
    "Plan a relaxed 2-day itinerary for Udaipur with lakes and palaces.",
]:
    print("=" * 55)
    print("INPUT:", c)
    r = request_app.invoke({"request": c, "request_type": "", "city_mentioned": "", "origin_mentioned": "",
                            "date_mentioned": "", "nights_mentioned": 2, "response": ""})
    done("RESPONSE: " + r["response"])
    print("")

INPUT: Book me a flight from Mumbai to Goa this Friday morning, cheapest option please.
  [READS]  request from state
  [WRITES] request_type=flight | city=Goa | origin=Mumbai | date=Friday | nights=2 → state
  [ROUTE]  request_type=flight → flight_specialist
  [READS]  request_type + city + origin + date from state
  [TOOL]   search_flights(Mumbai, Goa, Friday)
  [RESULT] Mumbai -> Goa on Friday: Cheapest Rs 4,113 (Budget Air, 07:15, 1h25m); Fastest Rs 4,863 (Prime Air, 09:40, 1h25m).
  [WRITES] response → state
  [OUTPUT] RESPONSE: FLIGHT BOOKING AGENT: Here is what I found: Mumbai -> Goa on Friday: Cheapest Rs 4,113 (Budget Air, 07:15, 1h25m); Fastest Rs 4,863 (Prime Air, 09:40, 1h25m). I can book the cheapest option for you right now, I only need the traveller's name.

INPUT: I need a hotel in Jaipur for 3 nights, mid-range with a good location.
  [READS]  request from state
  [WRITES] request_type=hotel | city=Jaipur | origin=Mumbai | date=Friday | nights=3 → state
  [ROUTE]  requ

---
## Agent 3 — Multi-City Trip Planner with Quality Gate

Plans 3 cities using real tool data. Scores quality. Loops until score >= 7.

In [8]:
class MultiCityState(TypedDict):
    cities: List[str]
    city_data: str
    city_plans: str
    quality: int
    attempts: int
    final_plan: str

def gather_node(state: MultiCityState) -> dict:
    agent("gather_node — calling tools for all cities")
    parts = []
    for city in state["cities"]:
        tool_c(f"get_destination_info({city})")
        info = get_destination_info.invoke({"city": city})
        result(info[:60])
        tool_c(f"check_availability({city}, Friday)")
        avail = check_availability.invoke({"city": city, "date": "Friday"})
        result(avail)
        parts.append(f"{city.upper()}: {info} | {avail}")
    writes("city_data → state")
    return {"city_data": "\n".join(parts)}

def create_plans_node(state: MultiCityState) -> dict:
    attempt = state["attempts"] + 1
    agent(f"create_plans_node — attempt {attempt}")
    reads("city_data from state")
    r = llm.invoke(
        "Based on this city data, write one-sentence plan for each city.\n\n"
        "Data:\n" + state["city_data"] + "\n\nFormat: CITY: specific plan."
    )
    writes("city_plans → state")
    return {"city_plans": r.content, "attempts": attempt}

def check_quality_node(state: MultiCityState) -> dict:
    agent("check_quality_node — scoring plans")
    reads("city_plans from state")
    r = llm.invoke("Rate these plans 1-10. ONE number only.\n\nPlans:\n" + state["city_plans"])
    try: q = int(r.content.strip())
    except: q = 6
    writes(f"quality={q}/10 → state")
    return {"quality": q}

def assemble_node(state: MultiCityState) -> dict:
    agent("assemble_node — building final plan")
    reads("city_plans + quality from state")
    final = (
        "MULTI-CITY TRIP PLAN — Friday departure\n" + "="*40 + "\n"
        "Cities  : " + ", ".join(state["cities"]) + "\n"
        "Quality : " + str(state["quality"]) + "/10\n"
        "Attempts: " + str(state["attempts"]) + "\n\n"
        "PLANS:\n" + state["city_plans"]
    )
    writes("final_plan → state")
    return {"final_plan": final}

def route_quality_m(state: MultiCityState) -> str:
    if state["quality"] >= 7:
        route(f"quality={state['quality']} >= 7 → PROCEED")
        return "assemble"
    if state["attempts"] >= 3:
        route("max attempts → assemble anyway")
        return "assemble"
    route(f"quality={state['quality']} < 7 → LOOP BACK")
    return "create_plans"

g3 = StateGraph(MultiCityState)
g3.add_node("gather",        gather_node)
g3.add_node("create_plans",  create_plans_node)
g3.add_node("check_quality", check_quality_node)
g3.add_node("assemble",      assemble_node)
g3.set_entry_point("gather")
g3.add_edge("gather",         "create_plans")
g3.add_edge("create_plans",   "check_quality")
g3.add_conditional_edges("check_quality", route_quality_m,
    {"assemble":"assemble","create_plans":"create_plans"})
g3.add_edge("assemble", END)
multi_app = g3.compile()

print("=" * 55)
print("Agent 3: Multi-City Trip Planner")
print("=" * 55)
mr = multi_app.invoke({"cities":["goa","kochi","udaipur"],"city_data":"","city_plans":"","quality":0,"attempts":0,"final_plan":""})
print("")
done("FINAL MULTI-CITY TRIP PLAN:")
print(mr["final_plan"])

Agent 3: Multi-City Trip Planner
  [AGENT]  gather_node — calling tools for all cities
  [TOOL]   get_destination_info(goa)
  [RESULT] Goa: best Nov-Feb, beaches + old Portuguese quarter, avg Rs 
  [TOOL]   check_availability(goa, Friday)
  [RESULT] 35 of 40 partner rooms available in goa on Friday.
  [TOOL]   get_destination_info(kochi)
  [RESULT] Kochi: best Oct-Feb, Fort Kochi + backwater cruises, avg Rs 
  [TOOL]   check_availability(kochi, Friday)
  [RESULT] 25 of 30 partner rooms available in kochi on Friday.
  [TOOL]   get_destination_info(udaipur)
  [RESULT] Udaipur: best Sep-Mar, City Palace + Lake Pichola, avg Rs 3,
  [TOOL]   check_availability(udaipur, Friday)
  [RESULT] 21 of 25 partner rooms available in udaipur on Friday.
  [WRITES] city_data → state
  [AGENT]  create_plans_node — attempt 1
  [READS]  city_data from state
  [WRITES] city_plans → state
  [AGENT]  check_quality_node — scoring plans
  [READS]  city_plans from state
  [WRITES] quality=8/10 → state
  [ROUTE] 

---
## Agent 4 — Trip Approval Pipeline

Research → Plan → Quality Loop → Human Pause → Finalise

Uses `MemorySaver` — works correctly with all langgraph versions.

> `SqliteSaver` can fail on some older langgraph versions (`SerializerCompat` has no attribute `dumps`).
> `MemorySaver` keeps the checkpoint in RAM and works the same way for human-in-the-loop workflows.

In [9]:
class TripApprovalState(TypedDict):
    cities: List[str]
    city_data: str
    draft_plan: str
    quality: int
    attempts: int
    approver_note: str
    final_plan: str

def research_node(state: TripApprovalState) -> dict:
    agent("research_node — collecting destination and hotel data with tools")
    parts = []
    for city in state["cities"]:
        tool_c(f"get_destination_info({city})")
        info = get_destination_info.invoke({"city": city})
        result(info[:60])
        tool_c(f"search_hotels({city}, 2)")
        hotels = search_hotels.invoke({"city": city, "nights": 2})
        result(hotels[:80])
        parts.append(f"{city.upper()}: {info} | {hotels}")
    writes("city_data → state")
    return {"city_data": "\n".join(parts)}

def draft_plan_node(state: TripApprovalState) -> dict:
    attempt = state["attempts"] + 1
    agent(f"draft_plan_node — creating plan (attempt {attempt})")
    reads("city_data from state")
    r = llm.invoke(
        "Create a concise travel plan for each city.\n\n"
        "City data:\n" + state["city_data"] + "\n\nFormat: CITY: one-sentence plan."
    )
    writes("draft_plan → state")
    return {"draft_plan": r.content, "attempts": attempt}

def quality_node(state: TripApprovalState) -> dict:
    agent("quality_node — scoring plan")
    reads("draft_plan from state")
    r = llm.invoke("Rate this plan 1-10. ONE number only.\n\nPlan:\n" + state["draft_plan"])
    try: q = int(r.content.strip())
    except: q = 6
    writes(f"quality={q}/10 → state")
    # Routing decision is printed here (not in route_q) because newer LangGraph versions
    # can re-run the routing function when the graph resumes after a pause.
    if q >= 7:                      route(f"quality={q} >= 7 → PROCEED")
    elif state["attempts"] >= 3:    route("max attempts → PROCEED anyway")
    else:                           route(f"quality={q} < 7 → LOOP BACK")
    return {"quality": q}

def final_node(state: TripApprovalState) -> dict:
    agent("final_node — assembling approved plan")
    reads("draft_plan + approver_note from state")
    plan = (
        "APPROVED TRIP PLAN\n" + "="*40 + "\n"
        "Cities  : " + ", ".join(state["cities"]) + "\n"
        "Quality : " + str(state["quality"]) + "/10\n"
        "Attempts: " + str(state["attempts"]) + "\n"
    )
    if state.get("approver_note"): plan += "Approver: " + state["approver_note"] + "\n"
    plan += "\nPLAN DETAILS:\n" + state["draft_plan"]
    writes("final_plan → state")
    return {"final_plan": plan}

def route_q(state: TripApprovalState) -> str:
    if state["quality"] >= 7 or state["attempts"] >= 3:
        return "final_node"
    return "draft_plan_node"

# MemorySaver keeps checkpoints in RAM — identical behaviour for human-in-the-loop
mem = MemorySaver()

g4 = StateGraph(TripApprovalState)
g4.add_node("research_node",   research_node)
g4.add_node("draft_plan_node", draft_plan_node)
g4.add_node("quality_node",    quality_node)
g4.add_node("final_node",      final_node)
g4.set_entry_point("research_node")
g4.add_edge("research_node",   "draft_plan_node")
g4.add_edge("draft_plan_node", "quality_node")
g4.add_conditional_edges("quality_node", route_q,
    {"final_node":"final_node","draft_plan_node":"draft_plan_node"})
g4.add_edge("final_node", END)
full_app = g4.compile(checkpointer=mem, interrupt_before=["final_node"])
config = {"configurable": {"thread_id": "trip-pipeline-001"}}

print("=" * 55)
print("Agent 4: Trip Approval Pipeline")
print("=" * 55)

full_app.invoke(
    {"cities":["jaipur","udaipur","kochi"],"city_data":"","draft_plan":"",
     "quality":0,"attempts":0,"approver_note":"","final_plan":""},
    config=config
)

paused = full_app.get_state(config)
print("")
pause("GRAPH PAUSED — Waiting for approver")
print("  Draft Plan:")
print("  " + paused.values["draft_plan"].replace("\n","\n  "))
print(f"  Quality: {paused.values['quality']}/10")
print(f"  Waiting at: {paused.next}")

Agent 4: Trip Approval Pipeline
  [AGENT]  research_node — collecting destination and hotel data with tools
  [TOOL]   get_destination_info(jaipur)
  [RESULT] Jaipur: best Oct-Mar, Amber Fort + City Palace + bazaars, av
  [TOOL]   search_hotels(jaipur, 2)
  [RESULT] Jaipur, 2 nights: Budget Rs 1,500/night (Rs 3,000 total) | Mid-range Rs 3,800/ni
  [TOOL]   get_destination_info(udaipur)
  [RESULT] Udaipur: best Sep-Mar, City Palace + Lake Pichola, avg Rs 3,
  [TOOL]   search_hotels(udaipur, 2)
  [RESULT] Udaipur, 2 nights: Budget Rs 2,000/night (Rs 4,000 total) | Mid-range Rs 4,800/n
  [TOOL]   get_destination_info(kochi)
  [RESULT] Kochi: best Oct-Feb, Fort Kochi + backwater cruises, avg Rs 
  [TOOL]   search_hotels(kochi, 2)
  [RESULT] Kochi, 2 nights: Budget Rs 1,700/night (Rs 3,400 total) | Mid-range Rs 4,000/nig
  [WRITES] city_data → state
  [AGENT]  draft_plan_node — creating plan (attempt 1)
  [READS]  city_data from state
  [WRITES] draft_plan → state
  [AGENT]  quality_node — 


  [PAUSE]  GRAPH PAUSED — Waiting for approver
  Draft Plan:
  JAIPUR: In the Oct-Mar season, stay at a mid-range hotel (Rs 3,800/night) and cover Amber Fort + City Palace + bazaars.
  UDAIPUR: In the Sep-Mar season, stay at a mid-range hotel (Rs 4,800/night) and cover City Palace + Lake Pichola.
  KOCHI: In the Oct-Feb season, stay at a mid-range hotel (Rs 4,000/night) and cover Fort Kochi + backwater cruises.
  Quality: 8/10
  Waiting at: ('final_node',)


In [10]:
human("Approver adding approval note to state...")
full_app.update_state(
    config,
    {"approver_note": "Approved. Upgrade the Udaipur stay to a lake-view room for the anniversary."}
)

agent("Resuming — final_node reads draft_plan + approver_note from state")
final_result = full_app.invoke(None, config=config)

print("")
done("FINAL APPROVED PLAN:")
print(final_result["final_plan"])

  [HUMAN]  Approver adding approval note to state...
  [AGENT]  Resuming — final_node reads draft_plan + approver_note from state
  [AGENT]  final_node — assembling approved plan
  [READS]  draft_plan + approver_note from state
  [WRITES] final_plan → state

  [OUTPUT] FINAL APPROVED PLAN:
APPROVED TRIP PLAN
Cities  : jaipur, udaipur, kochi
Quality : 8/10
Attempts: 1
Approver: Approved. Upgrade the Udaipur stay to a lake-view room for the anniversary.

PLAN DETAILS:
JAIPUR: In the Oct-Mar season, stay at a mid-range hotel (Rs 3,800/night) and cover Amber Fort + City Palace + bazaars.
UDAIPUR: In the Sep-Mar season, stay at a mid-range hotel (Rs 4,800/night) and cover City Palace + Lake Pichola.
KOCHI: In the Oct-Feb season, stay at a mid-range hotel (Rs 4,000/night) and cover Fort Kochi + backwater cruises.


---
## Summary

| Color | Meaning |
|---|---|
| Bold Blue | Agent thinking |
| Cyan | Tool called |
| Yellow | Tool result |
| Magenta | Agent writes to state |
| Blue | Agent reads from state |
| Dark Yellow | Routing decision |
| Bold Green | Final output |
| Bold Red | Graph paused |
| Bold Purple | Human action |

**Every agent finished → wrote to state → next agent picked up from state.**

**Next: Section 6.2 — AutoGen**